In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!unzip /content/drive/MyDrive/OMR-Datasets/OMR_5Fold_ROIs_split_v4.zip

Streaming output truncated to the last 5000 lines.
  inflating: content/OMR_5Fold_ROIs_split/Fold_5/val/empty/exam5_292_1_box48.jpg  
  inflating: content/OMR_5Fold_ROIs_split/Fold_5/val/empty/exam0_1_3_box3.jpg  
  inflating: content/OMR_5Fold_ROIs_split/Fold_5/val/empty/exam5_217_1_box21.jpg  
  inflating: content/OMR_5Fold_ROIs_split/Fold_5/val/empty/exam5_308_1_box42.jpg  
  inflating: content/OMR_5Fold_ROIs_split/Fold_5/val/empty/exam1_86_1_box11.jpg  
  inflating: content/OMR_5Fold_ROIs_split/Fold_5/val/empty/exam5_95_1_box32.jpg  
  inflating: content/OMR_5Fold_ROIs_split/Fold_5/val/empty/exam5_213_1_box34.jpg  
  inflating: content/OMR_5Fold_ROIs_split/Fold_5/val/empty/exam0_22_3_box9.jpg  
  inflating: content/OMR_5Fold_ROIs_split/Fold_5/val/empty/exam5_95_1_box21.jpg  
  inflating: content/OMR_5Fold_ROIs_split/Fold_5/val/empty/exam5_257_1_box57.jpg  
  inflating: content/OMR_5Fold_ROIs_split/Fold_5/val/empty/exam5_201_1_box5.jpg  
  inflating: content/OMR_5Fold_ROIs_split/Fol

In [4]:
import os
# train_Scen1_withoutGAN or train_Scen2_withGAN
len(os.listdir('/content/content/OMR_5Fold_ROIs_split/Fold_5/train_Scen2_withGAN/crossedout'))

2500

In [ ]:
import os
import copy
import time
import glob
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.transforms.functional as F
from torchvision import datasets, models, transforms
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, f1_score
from sklearn.metrics import confusion_matrix, classification_report
from torch.cuda.amp import autocast, GradScaler
from tqdm import tqdm

import warnings
warnings.filterwarnings("ignore", category=FutureWarning)


# ==========================================
# 1. CẤU HÌNH BIẾN ĐỔI ẢNH (BẬT CHỈNH SÁNG CHO TRAIN)
# ==========================================

class SquarePad:
    def __call__(self, image):
        w, h = image.size
        max_wh = np.max([w, h])
        hp = int((max_wh - w) / 2)
        vp = int((max_wh - h) / 2)
        padding = (hp, vp, hp, vp)
        # Đắp viền màu trắng (255, 255, 255) cho hợp với màu nền giấy thi
        return F.pad(image, padding, (255, 255, 255), 'constant')

data_transforms = {
    'train': transforms.Compose([
        SquarePad(),
        transforms.Resize((128, 128)),
        transforms.ColorJitter(brightness=0.3, contrast=0.3),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ]),
    'val': transforms.Compose([
        SquarePad(),
        transforms.Resize((128, 128)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ]),
    'test': transforms.Compose([
        SquarePad(),
        transforms.Resize((128, 128)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ]),
}

# Thư mục data
K_FOLDS_DIR = "/content/content/OMR_5Fold_ROIs_split"
# Thư mục lưu trọng số
WEIGHT_DIR = "/content/drive/MyDrive/OMR-Datasets/train-cls-v2/scene2/EfficientNetB0-v2"

# Đổi thành "train_Scen1_withoutGAN" or "train_Scen2_withGAN"
CHOSEN_SCENARIO = "train_Scen2_withGAN"
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")


# Hàm huấn luyện
def train_model(model, criterion, optimizer, scaler, dataloaders, device, fold, weight_folds, num_epochs=30, patience=7, use_amp=True):
    """
    Hàm huấn luyện mô hình với tiêu chí lưu mô hình tốt nhất dựa trên Macro F1-Score.
    """
    since = time.time()

    # Khởi tạo các biến lưu vết
    best_val_f1 = 0.0  # Thay đổi: Lưu best F1 thay vì best Acc
    best_model_wts = copy.deepcopy(model.state_dict())
    train_losses, val_losses = [], []
    train_f1s, val_f1s = [], [] # Lưu lịch sử F1
    counter = 0

    for epoch in range(num_epochs):
        # ==================== TRAIN ====================
        model.train()
        train_loss, train_correct, train_total = 0, 0, 0

        for images, labels in tqdm(dataloaders['train'], desc=f'Epoch {epoch+1}/{num_epochs} - Train'):
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad(set_to_none=True)

            with autocast(enabled=use_amp):
                outputs = model(images)
                loss = criterion(outputs, labels)

            if use_amp:
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()
            else:
                loss.backward()
                optimizer.step()

            train_loss += loss.item() * images.size(0)
            _, preds = torch.max(outputs, 1)
            train_correct += (preds == labels).sum().item()
            train_total += labels.size(0)

        train_loss /= train_total
        train_losses.append(train_loss)

        # ==================== VALIDATION ====================
        model.eval()
        val_loss = 0
        all_preds = []
        all_labels = []

        with torch.no_grad():
            for images, labels in tqdm(dataloaders['val'], desc='Validation'):
                images, labels = images.to(device), labels.to(device)

                with autocast(enabled=use_amp):
                    outputs = model(images)
                    loss = criterion(outputs, labels)

                val_loss += loss.item() * images.size(0)
                _, preds = torch.max(outputs, 1)

                # Gom kết quả để tính F1
                all_preds.extend(preds.cpu().numpy())
                all_labels.extend(labels.cpu().numpy())

        val_loss /= len(dataloaders['val'].dataset)
        val_losses.append(val_loss)

        # Tính Macro F1-Score
        val_f1 = f1_score(all_labels, all_preds, average='macro')
        val_f1s.append(val_f1)

        print(f'  => Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | Val F1-Score (Macro): {val_f1:.4f}')

        # ==================== LƯU MÔ HÌNH TỐT NHẤT (THEO F1) ====================
        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            best_model_wts = copy.deepcopy(model.state_dict())
            counter = 0
            os.makedirs(weight_folds, exist_ok=True)
            # Lưu model với F1-Score tốt nhất
            save_path = os.path.join(weight_folds, f'{weight_folds}/efficientnetb0_fold{fold}_gan.pth')
            torch.save(model.state_dict(), save_path)
            print(f"  🌟 Mô hình tốt hơn đã được lưu (Best Macro F1: {best_val_f1:.4f})")
        else:
            counter += 1
            if counter >= patience:
                print('  🛑 Early stopping triggered.')
                break

    time_elapsed = time.time() - since
    print(f'\n⏱️ Thời gian Train Fold {fold} hoàn tất: {time_elapsed // 60:.0f}p {time_elapsed % 60:.0f}s')
    print(f'🌟 Best Val F1-Score cho Fold {fold}: {best_val_f1:.4f}')

    model.load_state_dict(best_model_wts)
    history = {
        'train_loss': train_losses, 'val_loss': val_losses,
        'val_f1': val_f1s
    }
    return model, history


# ==========================================
# 2. VÒNG LẶP 5 FOLDS
# ==========================================
fold_results = {'acc': [], 'prec': [], 'rec': [], 'f1': []}
global_y_true = []
global_y_pred = []

for fold in range(1, 6):
    print(f"\n{'='*60}")
    print(f"🚀 BẮT ĐẦU HUẤN LUYỆN EFFICIENTNET - FOLD {fold} ({CHOSEN_SCENARIO})")
    print(f"{'='*60}")

    # Trỏ đường dẫn dữ liệu cho Fold hiện tại
    fold_dir = os.path.join(K_FOLDS_DIR, f"Fold_{fold}")
    train_dir = os.path.join(fold_dir, CHOSEN_SCENARIO)
    val_dir = os.path.join(fold_dir, "val")
    test_dir = os.path.join(fold_dir, "test")

    # đường dẫn lưu trọng số từng fold
    weight_folds = os.path.join(WEIGHT_DIR, f"Fold_{fold}")
    os.makedirs(weight_folds, exist_ok=True)

    image_datasets = {
        'train': datasets.ImageFolder(train_dir, data_transforms['train']),
        'val': datasets.ImageFolder(val_dir, data_transforms['val']),
        'test': datasets.ImageFolder(test_dir, data_transforms['test'])
    }

    dataloaders = {x: torch.utils.data.DataLoader(image_datasets[x], batch_size=64, shuffle=(x=='train'), num_workers=2)
                   for x in ['train', 'val', 'test']}

    class_names = image_datasets['train'].classes

    # ------------------------------------------
    # A. TÍNH TOÁN CLASS WEIGHTS CHO FOLD NÀY
    # ------------------------------------------
    class_counts = [0] * len(class_names)
    for _, label in image_datasets['train'].samples:
        class_counts[label] += 1

    total_samples = sum(class_counts)
    class_weights = [total_samples / (len(class_names) * count) for count in class_counts]
    class_weights_tensor = torch.FloatTensor(class_weights).to(device)
    print(f"📊 Phân bổ số lượng: {class_counts}")
    print(f"⚖️ Class Weights tự động: {class_weights}")

    # ------------------------------------------
    # B. KHỞI TẠO MÔ HÌNH MỚI (CHỐNG RÒ RỈ)
    # ------------------------------------------
    model = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.DEFAULT)
    num_ftrs = model.classifier[1].in_features
    model.classifier[1] = nn.Linear(num_ftrs, len(class_names))
    model = model.to(device)

    criterion = nn.CrossEntropyLoss(weight=class_weights_tensor)
    optimizer = optim.Adam(model.parameters(), lr=1e-4)

    use_amp = True
    scaler = GradScaler(enabled=use_amp)

    # ------------------------------------------
    # C. GỌI HÀM HUẤN LUYỆN
    # ------------------------------------------
    print("\n⏳ Đang tiến hành huấn luyện...")
    model, history = train_model(
        model=model,
        criterion=criterion,
        optimizer=optimizer,
        scaler=scaler,
        dataloaders=dataloaders,
        device=device,
        fold=fold,             # Truyền số thứ tự Fold vào để lưu file
        weight_folds=weight_folds,
        num_epochs=30,
        patience=10,            #
        use_amp=use_amp
    )

    # ------------------------------------------
    # D. ĐÁNH GIÁ TRÊN TẬP TEST (UNSEEN DATA)
    # ------------------------------------------
    print(f"\n🔍 ĐÁNH GIÁ TẬP TEST FOLD {fold}")
    model.eval()

    y_true = []
    y_pred = []

    with torch.no_grad():
        for inputs, labels in dataloaders['test']:
            inputs = inputs.to(device)
            labels = labels.to(device)
            outputs = model(inputs)
            _, preds = torch.max(outputs, 1)

            y_true.extend(labels.cpu().numpy())
            y_pred.extend(preds.cpu().numpy())

    # Tính các chỉ số
    acc = accuracy_score(y_true, y_pred)
    prec, rec, f1, _ = precision_recall_fscore_support(y_true, y_pred, average='macro', zero_division=0)

    fold_results['acc'].append(acc)
    fold_results['prec'].append(prec)
    fold_results['rec'].append(rec)
    fold_results['f1'].append(f1)

    print(f"✅ Fold {fold} | Acc: {acc:.4f} | F1: {f1:.4f}")

    # Gom dữ liệu để đánh giá Global
    global_y_true.extend(y_true)
    global_y_pred.extend(y_pred)

# ==========================================
# 3. TỔNG KẾT BÀI BÁO (AVERAGE ± STD)
# ==========================================
print(f"\n" + "="*60)
print(f"🏆 KẾT QUẢ 5-FOLD CROSS VALIDATION ({CHOSEN_SCENARIO})")
print("="*60)

# Hàm in định dạng đẹp
def print_metric(name, values):
    mean_val = np.mean(values) * 100
    std_val = np.std(values) * 100
    print(f"{name:<15}: {mean_val:.2f}% ± {std_val:.2f}%")


print("\nMa trận nhầm lẫn (Confusion Matrix):")
cm = confusion_matrix(global_y_true, global_y_pred)
print(cm)

print("\nBáo cáo chi tiết (Classification Report):")
report = classification_report(global_y_true, global_y_pred, target_names=class_names, digits=4)
print(report)

print_metric("Accuracy", fold_results['acc'])
print_metric("Precision", fold_results['prec'])
print_metric("Recall", fold_results['rec'])
print_metric("F1-Score", fold_results['f1'])
print("="*60)


🚀 BẮT ĐẦU HUẤN LUYỆN EFFICIENTNET - FOLD 1 (train_Scen2_withGAN)
📊 Phân bổ số lượng: [6605, 2500, 13484]
⚖️ Class Weights tự động: [1.1399949533181932, 3.0118666666666667, 0.5584149115000494]
Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b0_rwightman-7f5810bc.pth


100%|██████████| 20.5M/20.5M [00:00<00:00, 132MB/s]



⏳ Đang tiến hành huấn luyện...


Validation: 100%|██████████| 109/109 [00:11<00:00,  9.18it/s]


  => Train Loss: 0.1939 | Val Loss: 0.0366 | Val F1-Score (Macro): 0.8650
  🌟 Mô hình tốt hơn đã được lưu (Best Macro F1: 0.8650)


...

Validation: 100%|██████████| 106/106 [00:09<00:00, 10.97it/s]


  => Train Loss: 0.0044 | Val Loss: 0.0542 | Val F1-Score (Macro): 0.8730


Validation: 100%|██████████| 106/106 [00:09<00:00, 11.27it/s]

  => Train Loss: 0.0050 | Val Loss: 0.0454 | Val F1-Score (Macro): 0.8924

⏱️ Thời gian Train Fold 5 hoàn tất: 30p 38s
🌟 Best Val F1-Score cho Fold 5: 0.8985

🔍 ĐÁNH GIÁ TẬP TEST FOLD 5


✅ Fold 5 | Acc: 0.9974 | F1: 0.9458

🏆 KẾT QUẢ 5-FOLD CROSS VALIDATION (train_Scen2_withGAN)

Ma trận nhầm lẫn (Confusion Matrix):
[[10923    35    22]
 [   52   138    12]
 [   18    15 22334]]

Báo cáo chi tiết (Classification Report):
              precision    recall  f1-score   support

   confirmed     0.9936    0.9948    0.9942     10980
  crossedout     0.7340    0.6832    0.7077       202
       empty     0.9985    0.9985    0.9985     22367

    accuracy                         0.9954     33549
   macro avg     0.9087    0.8922    0.9001     33549
weighted avg     0.9953    0.9954    0.9953     33549

Accuracy       : 99.54% ± 0.16%
Precision      : 90.91% ± 4.14%
Recall         : 89.21% ± 3.09%
F1-Score       : 89.79% ± 2.77%
